In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import math

import numpy as np
import pandas as pd

import os
import os.path as osp

import matplotlib.pyplot as plt

from datasets import load_dataset, load_from_disk, ClassLabel, DatasetDict, Dataset
import pandas as pd

from dataloader import Loaders
from dataloader_10m import Loaders_10M

# 데이터셋 로드

In [2]:
# ds = load_dataset("nayohan/aihub-en-ko-translation-12m")
ds = load_from_disk("/home/paokimsiwoong/workspace/github.com/paokimsiwoong/transformer_experiments/eda/en_ko_12m")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['domain', 'subdomain', 'style', 'target', 'source', 'target_text', 'source_text'],
        num_rows: 11895711
    })
})


# 데이터셋 분석

In [3]:
unique_classes = ds.unique('domain')  
print(unique_classes)
print(f"==>> len(unique_classes): {len(unique_classes['train'])}")

{'train': ['일상생활', '해외고객과의채팅', '해외영업', '경제', '기술과학', '기후', '세계', '정치', '교양', '다큐', '연예공연', '영화드라마', '예능오락', '인터뷰', '기타', '다큐교양', '01000', '02000', '03000', '10110', '10121', '10122', '10129', '10210', '10220', '10300', '10400', '10500', '10610', '10620', '10712', '10713', '10720', '10730', '10742', '10743', '10749', '10750', '10791', '10792', '10795', '10796', '10797', '10799', '10801', '10802', '11110', '11120', '11200', '12000', '13103', '13104', '13109', '13213', '13219', '13221', '13224', '13229', '13400', '13920', '13992', '13994', '13999', '14120', '14191', '14192', '14194', '14199', '14411', '14419', '14491', '14499', '15129', '15211', '15219', '15220', '16101', '16102', '16103', '16211', '16212', '16221', '16229', '16230', '16300', '17110', '17120', '17220', '17901', '17902', '17909', '18110', '18120', '18200', '19210', '19221', '19229', '20111', '20119', '20121', '20129', '20131', '20132', '20201', '20202', '20203', '20311', '20312', '20313', '20321', '20322', '20411', '20413'

In [4]:
# 데이터셋 전체를 pandas로 바꾸지 않고, 데이터셋 형식만 변경하여 확인 (메모리 절약)
ds.set_format(type="pandas")
df = ds['train']
print(df['domain'].value_counts())


domain
None                             1501772
해외영업                             1260367
일상생활                              899757
경제                                548114
해외고객과의채팅                          540221
                                  ...   
역사/근현대|문화·교육/언론·출판|문화유산/기록 유산          1
역사/근현대|종교/기독교|성씨·인물/근현대인물              1
역사/근현대|정치·경제·사회/경제 산업                  1
역사/근현대|인물/근현대 인물                       1
정치·경제·사회/경제·산업|문화·교육/문화·예술             1
Name: count, Length: 664, dtype: int64


In [5]:
print(df['style'].value_counts())

style
문어체     5923098
구어체     3119332
None    2393282
특허       359999
대화체      100000
Name: count, dtype: int64


In [6]:
print(df['subdomain'].value_counts())

subdomain
None                  3915439
도소매유통                 1025979
음식                     314915
여행                     291357
법률연구                   244865
                       ...   
농산물                         3
인물 사진 및 행사용 영상 촬영업          2
식품일반                        2
보이스7                        2
건강식품류                       1
Name: count, Length: 657, dtype: int64


In [7]:

# 자료 확인 후 ds 원래 포맷으로 복구
ds.reset_format()

TODO: 데이터셋 처음 상태에서 다양한 분석 결과 추가하기

# 대분류 추가

In [8]:
count = 0

def map_domain(raw_domain: str) -> str:
    if raw_domain is None:
        return "기타"

    d = raw_domain

    # 1) 명시적인 매핑
    if d in ["일상생활", "구어체_대화체", "해외고객과의채팅", "해외영업"]:
        return "일상/대화"

    if d in ["뉴스문어체", "지자체웹사이트 문어체", "가정통신문", "스포츠"]:
        return "뉴스/시사"

    if d in ["교양", "다큐", "다큐교양", "연예공연", "영화드라마", "예능오락", "인터뷰", "기타", # 방송콘텐츠 데이터셋에 기타 분류가 있음
             "문화문어체", 
             "문화", "문화·예술", "문화·교육", "문화재", "민속", "생활·민속",
             "문화유산", "역사", "역사/근현대", "역사/전통 시대",
             "구비 전승·언어·문학", "성씨·인물",
             "관광", "예술", "향토문화/음식", "종교", "지리",
             "정치·경제·사회", "정치∙경제∙사회", # 종류가 두가지?
             "정치·경제·산업/경제·산업|역사/근현대", "한국국제문화교류진흥원"]:
        return "문화/예술/역사"
    

    if d in ["경제", "세계", "정치", "기후", "기술과학", # 기술과학 분야 한-영 번역 병렬 말뭉치 데이터
             "글로벌동향정보", "연구평가정보", "위해식품정보", "법제도정보", # 식품 전문 분야 데이터
             "IT/기술", "ICT", "컴퓨터과학", "정보-통신",
             "공학", "전기", "전자", "재료", "재료과학",
             "수학", "물리학", "미생물학", "화학", "생명과학", "생물학 생화학",
             "환경과 생태학", "농학", "농림수산식품", "기계",
             "사회", "사회과학", "전문분야 문어체", "교육"]:
        return "과학/기술/학술자료"

    if d in ["의료/보건", "보건의료", "의학", "의약학", "의약학", "약리학 독성학"]:
        return "의학/보건"

    if d in ["법률", "대법원판례", "조례문어체", "교통"]:
        return "법률/행정"

    if d in ["금융/증시"]:
        return "금융/경제"

    # 2) 패턴 기반 매핑 (복잡한 문자열들)
    if d.startswith("문화·교육") or d.startswith("문화유산") or d.startswith("생활·민속"):
        return "문화/예술/역사"
    if d.startswith("역사/"):
        return "문화/예술/역사"
    if d.startswith("종교/"):
        return "문화/예술/역사"
    if d.startswith("지리/"):
        return "문화/예술/역사"
    if d.startswith("정치·경제·사회"):
        return "문화/예술/역사"
    if "의료" in d or "의학" in d or "보건" in d:
        return "의학/보건"
    
    # 특허
    if d.isdigit():
        return "특허"
    
    if d in ["H", "K"]:
        return "특허"

    # 3) 그 밖의 코드/기타 값
    if d in ["None"]:
        return "기타"

    # 조건에서 벗어난 경우의 수 카운트
    global count 
    count += 1
    print(d)
    # 기본값
    return "기타"


In [9]:
def map_domain_batch(batch):
    batch["super_domain"] = [map_domain(d) for d in batch["domain"]]
    return batch

In [10]:
ds = ds.map(map_domain_batch, batched=True)

In [11]:
ds.set_format(type="pandas")
df = ds['train']

In [12]:
print(df['super_domain'].value_counts())

super_domain
과학/기술/학술자료    3822044
일상/대화         2719332
기타            1501772
문화/예술/역사      1346542
의학/보건          720037
법률/행정          626520
뉴스/시사          619465
특허             359999
금융/경제          180000
Name: count, dtype: int64


In [13]:

# 자료 확인 후 ds 원래 포맷으로 복구
ds.reset_format()

# 기타 대분류 1501772개 == AI HUB 한국어-영어 번역(병렬) 말뭉치 데이터셋

In [14]:
df = ds['train'].to_pandas()

In [15]:
df_etc = df[df['super_domain'] == "기타"]

In [16]:
df_etc["domain"].value_counts()

domain
None    1501772
Name: count, dtype: int64

In [17]:
df_etc["subdomain"].value_counts()

subdomain
None    1501772
Name: count, dtype: int64

In [18]:
df_etc["style"].value_counts()

style
문어체    1001772
구어체     400000
대화체     100000
Name: count, dtype: int64

In [19]:
df_aihub = pd.read_csv("data.csv")
len(df_aihub)

1602418

In [ ]:
merged = df_etc.merge(
    df_aihub.rename(columns={"en": "source_text", "kor": "target_text"}),
    on=["source_text", "target_text"],
    how="outer",
    indicator=True
)

print("df_etc에도 df_aihub에도 있는 행들:")
print(merged[merged["_merge"] == "both"].shape[0])

print("df_etc에만 있는 행들:")
print(merged[merged["_merge"] == "left_only"].shape[0])

print("df_aihub에만 있는 행들:")
print(merged[merged["_merge"] == "right_only"].shape[0])

df_etc에서 df_aihub에도 있는 행들:
1504288
df_etc에만 있는 행들:
0
df_aihub에만 있는 행들:
98174


# df_etc 길이는 1501722인데 both가 1504288로 더 큼 (2516 차이) => 중복 문장 쌍 개수 확인

In [21]:
# df_aihub에서 중복된 (en, kor) 쌍이 몇 개인지
pairs_aihub = df_aihub[["en", "kor"]].value_counts()
duplicated_pairs = pairs_aihub[pairs_aihub > 1]
print(f"df_aihub 중복 쌍 개수: {len(duplicated_pairs)}")
print(f"중복 쌍으로 인한 추가 행 수: {duplicated_pairs.sum() - len(duplicated_pairs)}")


df_aihub 중복 쌍 개수: 2494
중복 쌍으로 인한 추가 행 수: 2494


In [22]:
# df_etc에서 중복된 (en, kor) 쌍이 몇 개인지
pairs_etc = df_etc[["source_text", "target_text"]].value_counts()
duplicated_pairs = pairs_etc[pairs_etc > 1]
print(f"df_etc 중복 쌍 개수: {len(duplicated_pairs)}")
print(f"중복 쌍으로 인한 추가 행 수: {duplicated_pairs.sum() - len(duplicated_pairs)}")


df_etc 중복 쌍 개수: 22
중복 쌍으로 인한 추가 행 수: 22


2494 + 22 == 2516 ==> 확인 완료

# 중복 제거 후 확인

In [23]:
# df_etc에서 (source_text, target_text) 쌍 기준으로 중복 제거
df_etc_unique = df_etc.drop_duplicates(subset=["source_text", "target_text"])
print(f"==>> len(df_etc_unique): {len(df_etc_unique)}")

# df_aihub에서 (en, kor) 쌍 기준으로 중복 제거
df_aihub_unique = df_aihub.drop_duplicates(subset=["en", "kor"])
print(f"==>> len(df_aihub_unique): {len(df_aihub_unique)}")



==>> len(df_etc_unique): 1501750
==>> len(df_aihub_unique): 1599924


In [ ]:
merged_unique = df_etc_unique.merge(
    df_aihub_unique.rename(columns={"en": "source_text", "kor": "target_text"}),
    on=["source_text", "target_text"],
    how="outer",
    indicator=True
)

print("df_etc_unique에도 df_aihub_unique에도 있는 행들:")
print(merged_unique[merged_unique["_merge"] == "both"].shape[0])

print("df_etc_unique에만 있는 행들:")
print(merged_unique[merged_unique["_merge"] == "left_only"].shape[0])

print("df_aihub_unique에만 있는 행들:")
print(merged_unique[merged_unique["_merge"] == "right_only"].shape[0])

df_etc_unique에서 df_aihub_unique에도 있는 행들:
1501750
df_etc_unique에만 있는 행들:
0
df_aihub_unique에만 있는 행들:
98174


# AI HUB 한국어-영어 번역(병렬) 말뭉치 데이터셋 (1602418) - 기타 (1501772) == 100646
- 100646은 AI HUB 한국어-영어 번역(병렬) 말뭉치 데이터셋의 문어체_한국문화(cat=3) 데이터 개수와 동일
- 문어체_한국문화 분류의 데이터가 12m 데이터셋의 기타 제외 부분에 포함되어 있는지 확인하기

In [25]:
df_aihub_3 = df_aihub[df_aihub["cat"] == 3]
print(f"==>> len(df_aihub_3): {len(df_aihub_3)}")

==>> len(df_aihub_3): 100646


In [26]:
df_not_etc =  df[df['super_domain'] != "기타"]
print(f"==>> len(df_not_etc): {len(df_not_etc)}")

==>> len(df_not_etc): 10393939


In [27]:
# df_not_etc에서 (source_text, target_text) 쌍 기준으로 중복 제거
df_not_etc_unique = df_not_etc.drop_duplicates(subset=["source_text", "target_text"])
print(f"==>> len(df_not_etc_unique): {len(df_not_etc_unique)}")

# df_aihub에서 (en, kor) 쌍 기준으로 중복 제거
df_aihub_3_unique = df_aihub_3.drop_duplicates(subset=["en", "kor"])
print(f"==>> len(df_aihub_3_unique): {len(df_aihub_3_unique)}")



==>> len(df_not_etc_unique): 9734537
==>> len(df_aihub_3_unique): 100646


In [ ]:
df_not_etc_unique.to_csv("preprocess/not_etc_unique.csv")

: 

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import math

import numpy as np
import pandas as pd

import os
import os.path as osp

import matplotlib.pyplot as plt

from datasets import load_dataset, load_from_disk, ClassLabel, DatasetDict, Dataset
import pandas as pd

from dataloader import Loaders
from dataloader_10m import Loaders_10M

In [2]:
df_not_etc_unique = pd.read_csv("preprocess/not_etc_unique.csv")
print(f"==>> len(df_not_etc_unique): {len(df_not_etc_unique)}")

/tmp/ipykernel_21740/1068616865.py:1: DtypeWarning: Columns (1,2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_not_etc_unique = pd.read_csv("preprocess/not_etc_unique.csv")


==>> len(df_not_etc_unique): 9734537


In [3]:
df_aihub = pd.read_csv("data.csv")
df_aihub_3 = df_aihub[df_aihub["cat"] == 3]
print(f"==>> len(df_aihub_3): {len(df_aihub_3)}")

==>> len(df_aihub_3): 100646


In [4]:
merged_unique = df_not_etc_unique.merge(
    df_aihub_3.rename(columns={"en": "source_text", "kor": "target_text"}),
    on=["source_text", "target_text"],
    how="outer",
    indicator=True
)

print("df_not_etc_unique에도 df_aihub_3에도 있는 행들:")
print(merged_unique[merged_unique["_merge"] == "both"].shape[0])

print("df_not_etc_unique에만 있는 행들:")
print(merged_unique[merged_unique["_merge"] == "left_only"].shape[0])

print("df_aihub_3에만 있는 행들:")
print(merged_unique[merged_unique["_merge"] == "right_only"].shape[0])

df_not_etc_unique에서 df_aihub_3에도 있는 행들:
100646
df_not_etc_unique에만 있는 행들:
9633891
df_aihub_3에만 있는 행들:
0


In [5]:
both = merged_unique[merged_unique["_merge"] == "both"]
both["super_domain"].value_counts()


super_domain
문화/예술/역사    100158
뉴스/시사          477
법률/행정           11
Name: count, dtype: int64

In [6]:
both["style"].value_counts()

style
문어체    100646
Name: count, dtype: int64

AI HUB 한국어-영어 번역(병렬) 말뭉치 데이터셋의 문어체_한국문화도 12m 데이터셋에 포함되어 있는 것을 확인 가능
12m 데이터셋에 super_domain 기타로 분류되어 있던 다른 분류 데이터들과 다르게 문어체_한국문화 분류의 데이터는 문화/예술/역사, 뉴스/시사, 법률/행정 등으로 분류